- https://yingru.notion.site/When-Speed-Kills-Stability-Demystifying-RL-Collapse-from-the-Training-Inference-Mismatch-271211a558b7808d8b12d403fd15edda
    - https://verl.readthedocs.io/en/latest/algo/rollout_corr.html
- 所谓的训推一致，不仅仅是模型权重一样。它是指同一套参数，在 Megatron（训练框架）和 SGLang（推理框架）中前向传播（Forward）出来的 Log Probability 必须完全一致。
    - 很多时候，训练时采样参数（Temperature）是 0.8，但部署上线时默认参数变成了 1.0。这种微小的配置差异，或者 Tokenizer 处理上的细微不同，都会导致效果天差地别。

### Inference-Training Mismatch

采样（Rollout）阶段和训练（Update）阶段在计算概率时产生的数值差异 。
- Rollout (Inference) 阶段： 为了追求极高的吞吐量和并发（K2.5 支持 10 万并发任务 ），通常使用高度优化的推理引擎（如 vLLM, TensorRT-LLM，或 Kimi 自研的推理引擎）。这些引擎可能使用了 FP8/Int8 量化、特定的 Kernel 优化或近似计算（如 FlashAttention 的不同实现）。
- Update (Training) 阶段： 为了保证梯度更新的精度，训练通常在 PyTorch/Megatron 等框架下进行，使用 BF16 或 FP32。
- 后果： 即使是同一个模型权重，输入相同的 Context，推理引擎和训练引擎计算出的 Log Probabilities (logprobs) 也会有细微的差异。在长思维链（Long CoT）中，这种微小的数值误差会随着序列长度累积，导致 $\pi_{\text{old}}$（推理时记录的概率）和 $\pi_{\theta}$（训练时重新计算的概率）之间出现非算法本身导致的偏差。

### k2.5: Token-level Clipping

$$
\text{Clip} \left( \frac{\pi_{\theta}(y_j^i|...)}{\pi_{\text{old}}(y_j^i|...)}, \alpha, \beta \right)
$$

- 过滤“虚假”的 Off-policy 信号：当推理和训练框架存在 mismatch 时，可能会出现一种极端情况：推理引擎认为某个 Token 的概率是 0.1，而训练引擎计算出它是 0.9（反之亦然）。这会导致重要性采样比率（Ratio $\rho_t = \pi_\theta / \pi_{old}$）出现剧烈的、非真实的爆炸。如果直接使用这个 Ratio 更新，模型会误以为这是一个巨大的“惊喜”信号，从而进行大幅度的参数更新。但实际上，这只是工程误差。
- K2.5 的做法： 只要 Ratio 超出 $[\alpha, \beta]$ 的范围（例如 $[0.9, 1.1]$），该 Token 对应的梯度直接归零（Zeroed out） 。这意味着模型选择**“忽略”**那些差异过大的点，而不是试图去学习它们。
- 独立于 Advantage 的符号：报告特别指出，这种机制与标准 PPO 的 Clipping 不同。标准 PPO 依赖 Advantage 的正负号来决定是否截断，而 K2.5 的方法仅依赖 Log-ratio。这意味着，无论这个 Token 是“好”还是“坏”，只要计算出的概率偏移过大（哪怕是因为框架误差导致的），就强制停止更新。这极大地提升了训练的数值稳定性。
- 长思维链（Long CoT）场景下的特殊必要性：一个长 Chain 可能包含数千个 Token。如果不加限制，框架 Mismatch 导致的微小误差会在序列末端被指数级放大。
    - 公式是在 $\sum \text{Clip}(...)$ 层面运作的，也就是对每一个 Token 独立进行体检。如果某一步推理因为框架误差导致概率比率异常，就只丢弃那一两步的梯度，而保留其他正常步骤的梯度。